# Evaluación del Modelo — DocShield

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_curve, precision_recall_curve, 
                             roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix)
import xgboost as xgb
import lightgbm as lgb
import sys
sys.path.append('..')

from src.model.trainer import FEATURE_COLS

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
df = pd.read_parquet('../data/gold/dataset.parquet')
X = df[FEATURE_COLS]
y = df['is_fraud']

print(f'Dataset: {len(df)} muestras')
print(f'Clase positiva: {y.sum()} ({y.mean()*100:.1f}%)')

In [ ]:
# Entrenar XGBoost
print('Entrenando XGBoost...')
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                              objective='binary:logistic', eval_metric='aucpr',
                              random_state=42, n_jobs=-1)
xgb_model.fit(X, y)

# Entrenar LightGBM
print('Entrenando LightGBM...')
lgb_model = lgb.LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                              objective='binary', metric='auc',
                              random_state=42, n_jobs=-1, verbose=-1)
lgb_model.fit(X, y)

In [ ]:
# Evaluación con StratifiedKFold
def evaluate_model_cv(model, X, y, cv=5):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    y_pred_proba = cross_val_predict(model, X, y, cv=skf, method='predict_proba')[:, 1]
    
    roc_auc = roc_auc_score(y, y_pred_proba)
    pr_auc = average_precision_score(y, y_pred_proba)
    
    return roc_auc, pr_auc, y_pred_proba

print('Evaluando XGBoost...')
xgb_roc, xgb_pr, xgb_proba = evaluate_model_cv(xgb_model, X, y)
print(f'XGBoost - ROC-AUC: {xgb_roc:.4f}, PR-AUC: {xgb_pr:.4f}')

print('\nEvaluando LightGBM...')
lgb_roc, lgb_pr, lgb_proba = evaluate_model_cv(lgb_model, X, y)
print(f'LightGBM - ROC-AUC: {lgb_roc:.4f}, PR-AUC: {lgb_pr:.4f}')

In [ ]:
# Curvas ROC
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC
fpr_xgb, tpr_xgb, _ = roc_curve(y, xgb_proba)
fpr_lgb, tpr_lgb, _ = roc_curve(y, lgb_proba)

axes[0].plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC={xgb_roc:.3f})', lw=2)
axes[0].plot(fpr_lgb, tpr_lgb, label=f'LightGBM (AUC={lgb_roc:.3f})', lw=2)
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PR Curve
prec_xgb, rec_xgb, _ = precision_recall_curve(y, xgb_proba)
prec_lgb, rec_lgb, _ = precision_recall_curve(y, lgb_proba)

axes[1].plot(rec_xgb, prec_xgb, label=f'XGBoost (AP={xgb_pr:.3f})', lw=2)
axes[1].plot(rec_lgb, prec_lgb, label=f'LightGBM (AP={lgb_pr:.3f})', lw=2)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Matriz de confusión (usando threshold 0.5)
from sklearn.metrics import confusion_matrix
import seaborn as sns

y_pred = (xgb_proba >= 0.5).astype(int)
cm = confusion_matrix(y, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Pred Legit', 'Pred Fraud'],
            yticklabels=['True Legit', 'True Fraud'])
plt.title('Confusion Matrix (XGBoost, threshold=0.5)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print('\nReporte de clasificación:')
print(classification_report(y, y_pred, target_names=['Legit', 'Fraud']))

In [ ]:
# Comparación threshold vs métricas
thresholds = np.arange(0.1, 0.9, 0.05)
recall_scores = []
precision_scores = []

for thresh in thresholds:
    y_pred_thresh = (xgb_proba >= thresh).astype(int)
    from sklearn.metrics import recall_score, precision_score
    recall_scores.append(recall_score(y, y_pred_thresh))
    precision_scores.append(precision_score(y, y_pred_thresh))

plt.figure(figsize=(10, 6))
plt.plot(thresholds, recall_scores, label='Recall', lw=2)
plt.plot(thresholds, precision_scores, label='Precision', lw=2)
plt.axvline(x=0.35, color='r', linestyle='--', label='Umbral DocShield (0.35)')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision y Recall vs Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()